# BƯỚC 6: TRIỂN KHAI VÀ DỰ BÁO THỰC TẾ (INFERENCE)

**Mục tiêu:** Áp dụng mô hình Học máy đã huấn luyện để đưa ra dự báo xu hướng giá cổ phiếu (Tăng/Giảm) cho ngày giao dịch tiếp theo.

**Quy trình Pipeline triển khai:**
1. Nạp dữ liệu mới nhất (đã được chuẩn hóa từ Bước 2).
2. Tái tạo lại toàn bộ các Đặc trưng (Feature Engineering - Bước 3) cho dòng dữ liệu mới nhất này.
3. Tự động tìm và nạp mô hình có hiệu suất cao nhất (từ Bước 4, 5).
4. Tiến hành dự báo và xuất ra Khuyến nghị hành động (Mua/Nắm giữ hoặc Bán/Đứng ngoài).

In [ ]:
import os
import numpy as np
import pandas as pd
import pickle
from sklearn.metrics import f1_score
import warnings

warnings.filterwarnings('ignore')

print("--- 6.1: Khởi tạo Pipeline Tiền xử lý ---")

def create_features_inference(df):
    df_feat = df.copy()
    
    #Làm sạch tên cột
    df_feat.columns = df_feat.columns.str.strip()
    
    #Lag Features
    for i in [1, 2, 3]:
        df_feat[f'Close_Lag_{i}'] = df_feat['Close'].shift(i)
        df_feat[f'Volume_Lag_{i}'] = df_feat['Volume'].shift(i)
        
    #Moving Averages
    df_feat['SMA_10'] = df_feat['Close'].rolling(window=10).mean()
    df_feat['SMA_30'] = df_feat['Close'].rolling(window=30).mean()
    df_feat['EMA_10'] = df_feat['Close'].ewm(span=10, adjust=False).mean()
    
    #RSI
    delta = df_feat['Close'].diff()
    gain = delta.where(delta > 0, 0).rolling(window=14).mean()
    loss = -delta.where(delta < 0, 0).rolling(window=14).mean()
    rs = gain / loss
    df_feat['RSI_14'] = 100 - (100 / (1 + rs))
    df_feat['RSI_14'] = df_feat['RSI_14'] / 100.0
    
    #MACD
    ema_12 = df_feat['Close'].ewm(span=12, adjust=False).mean()
    ema_26 = df_feat['Close'].ewm(span=26, adjust=False).mean()
    df_feat['MACD'] = ema_12 - ema_26
    df_feat['MACD_Signal'] = df_feat['MACD'].ewm(span=9, adjust=False).mean()

    #Bollinger Bands
    bb_mid = df_feat['Close'].rolling(window=20).mean()
    bb_std = df_feat['Close'].rolling(window=20).std()
    
    df_feat['BB_Mid'] = bb_mid
    df_feat['BB_Upper'] = bb_mid + (bb_std * 2)
    df_feat['BB_Lower'] = bb_mid - (bb_std * 2)
    
    #Các đặc trưng khác
    df_feat['Day_Of_Week'] = df_feat.index.dayofweek 
    df_feat['Return_1D'] = df_feat['Close'].pct_change()
    df_feat['Volume_Change'] = df_feat['Volume'].pct_change()
    df_feat['Price_to_SMA30'] = df_feat['Close'] / df_feat['SMA_30']
    
    #Xử lý các cột tham chiếu an toàn bằng if-in
    if 'VNINDEX_Close' in df_feat.columns:
        df_feat['VNINDEX_Return'] = df_feat['VNINDEX_Close'].pct_change()
        
    if 'USDVND_Close' in df_feat.columns:
        df_feat['USDVND_Change'] = df_feat['USDVND_Close'].pct_change()
        
    #Xử lý giá trị vô cực (Infinity)
    df_feat.replace([np.inf, -np.inf], np.nan, inplace=True)
    
    return df_feat

print("-> Đã nạp thành công hàm create_features_inference()")

## 6.2. Tiến hành Dự báo (Predict Tomorrow)
Hệ thống sẽ trích xuất đúng dòng dữ liệu của ngày cuối cùng, đưa vào mô hình tốt nhất để quét tín hiệu và đưa ra quyết định giao dịch cho ngày mai.

In [ ]:
print("--- 6.2: Chạy dự báo (Inference) cho phiên giao dịch tiếp theo ---")

target_tickers = ['VNM', 'FPT', 'VIC']
model_names = ['logistic_regression', 'random_forest', 'xgboost']
split_date = '2024-12-31'

processed_dir = '../data/processed'
model_dir = '../models'

for ticker in target_tickers:
    print(f"\n[INFERENCE] Đang xử lý mã: {ticker}")
    
    #Xác định mô hình tốt nhất
    test_path = os.path.join(processed_dir, f"{ticker}_features.csv")
    if not os.path.exists(test_path):
        print(f"-> Bỏ qua {ticker}: Thiếu dữ liệu test.")
        continue
        
    #Nạp data và lọc nhanh tập Test
    df_test = pd.read_csv(test_path, index_col='Date', parse_dates=True)
    test_mask = df_test.index > split_date
    X_test = df_test[test_mask].drop(columns=['Target_Class'])
    y_test = df_test[test_mask]['Target_Class']
    
    best_f1 = 0
    best_model = None
    best_model_name = ""
    
    #Quét model nào có sẵn và đánh giá nhanh trên tập Test để chọn ra model tốt nhất (theo F1-Macro)
    for m_name in model_names:
        model_path = os.path.join(model_dir, f"{m_name}_{ticker}.pkl")
        if os.path.exists(model_path):
            with open(model_path, 'rb') as f:
                model = pickle.load(f)
                
            f1 = f1_score(y_test, model.predict(X_test), average='macro')
            if f1 > best_f1:
                best_f1, best_model, best_model_name = f1, model, m_name
                
    if not best_model:
        print(f"-> Bỏ qua {ticker}: Không tìm thấy file model (.pkl)")
        continue

    #Nạp dữ liệu mới nhất & Tạo đặc trưng
    scaled_path = os.path.join(processed_dir, f"{ticker}_scaled.csv")
    df_scaled = pd.read_csv(scaled_path, index_col='Date', parse_dates=True)
    
    #SỬ DỤNG HÀM INFERENCE
    df_infer = create_features_inference(df_scaled)
    
    #Trích xuất dòng cuối cùng (ngày giao dịch mới nhất)
    latest_date = df_infer.index[-1].strftime('%Y-%m-%d')
    
    #Khớp chính xác thứ tự cột với lúc Train để tránh lỗi Feature Mismatch
    X_latest = df_infer.iloc[[-1]][X_test.columns]
    
    #Tiến hành dự báo
    prediction = best_model.predict(X_latest)[0]
    
    #Xuất tín hiệu giao dịch ra Terminal
    print(f" -> Ngày chốt data : {latest_date}")
    print(f" -> Model sử dụng  : {best_model_name.upper()} (F1-Macro: {best_f1:.4f})")
    
    if prediction == 1:
        print(" -> TÍN HIỆU       : [TĂNG]")
        print(" -> HÀNH ĐỘNG      : Khuyến nghị MUA (Buy) hoặc NẮM GIỮ (Hold)")
    else:
        print(" -> TÍN HIỆU       : [GIẢM / ĐI NGANG]")
        print(" -> HÀNH ĐỘNG      : Khuyến nghị BÁN (Sell) hoặc ĐỨNG NGOÀI (Wait)")